# Video → GVHMR → Roblox R15 animation  (Colab, T4 GPU)

**Runtime → Change runtime type → T4 GPU**, then run cells top to bottom.

**How this notebook avoids dependency hell:** GVHMR needs Python 3.10 + torch 2.3 + numpy 1.23
(its pytorch3d wheel only exists for cp310). Colab's kernel is Python 3.12/3.13 + numpy 2.
So we **don't touch the Colab kernel at all**: we build a separate Python 3.10 virtualenv at
`/content/gvenv` with `uv`, and run every GVHMR command with `/content/gvenv/bin/python`.
No kernel restarts, and nothing Colab pre-installed can bump numpy.

Each setup cell ends with a ✅/❌ check. If you get a ❌, send the cell's full output.

### 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 2. Install (one reproducible block, ~5–8 min)
Why the old install broke: GVHMR's `setup.py` has **no dependencies listed**, so `pip install -e .`
installs nothing. The real list is `requirements.txt`. We install that into the 3.10 venv with
numpy **constrained** to 1.23.5 for every step (so the resolver picks compatible versions of
opencv/scikit-image/etc. instead of upgrading numpy), and re-pin numpy at the very end.

In [ ]:
%%bash
set -euo pipefail
cd /content
# Check for the code itself (.git), not just the folder: weight cells can recreate GVHMR/inputs alone.
if [ ! -d GVHMR/.git ]; then
  rm -rf /content/GVHMR_src
  git clone --recursive https://github.com/zju3dv/GVHMR.git /content/GVHMR_src
  mkdir -p GVHMR && cp -rn /content/GVHMR_src/. GVHMR/ && rm -rf /content/GVHMR_src   # keeps existing inputs/
fi
cd GVHMR
mkdir -p inputs/checkpoints outputs

pip install -q uv
[ -x /content/gvenv/bin/python ] || uv venv /content/gvenv --python 3.10 --seed
PY=/content/gvenv/bin/python

printf "numpy==1.23.5
setuptools<70
" > /content/constraints.txt   # lightning 2.3 needs pkg_resources (removed in new setuptools)
# Drop dev-only tools; chumpy + cython_bbox need --no-build-isolation, done separately below.
grep -v -E '^(jupyter|ipdb|black|chumpy|cython_bbox)([ =#]|$)' requirements.txt > /content/req_colab.txt

echo ">>> main requirements (torch 2.3.0+cu121, pytorch3d, lightning, ultralytics, ...)"
uv pip install --python $PY --index-strategy unsafe-best-match \
    -c /content/constraints.txt -r /content/req_colab.txt "pytorch-lightning==2.3.0"

echo ">>> build-time deps, then chumpy + cython_bbox"
uv pip install --python $PY -c /content/constraints.txt cython setuptools wheel
uv pip install --python $PY --no-build-isolation -c /content/constraints.txt chumpy cython_bbox \
  || uv pip install --python $PY --no-build-isolation -c /content/constraints.txt \
       "chumpy @ git+https://github.com/mattloper/chumpy" cython_bbox

echo ">>> GVHMR itself (no deps, it declares none anyway)"
uv pip install --python $PY --no-deps -e .

echo ">>> numpy pinned LAST"
uv pip install --python $PY "numpy==1.23.5" "setuptools>=68,<70"
echo "install finished"

In [ ]:
%%bash
# Verify the venv. Runs in the venv's Python 3.10, not the Colab kernel.
export MPLBACKEND=Agg PYTHONPATH=/content/GVHMR   # Colab's inline matplotlib backend doesn't exist in the venv
cd /content/GVHMR && /content/gvenv/bin/python - <<'EOF'
import sys, importlib
ok = True
print("python", sys.version.split()[0])
for m in ["numpy","torch","torchvision","pytorch3d","pytorch_lightning","lightning","hydra",
          "hydra_zen","colorlog","cv2","ultralytics","cython_bbox","lap",
          "chumpy","smplx","timm","einops","av","imageio","skimage","hmr4d"]:
    try:
        mod = importlib.import_module(m); print(f"  ok  {m:18s} {getattr(mod,'__version__','')}")
    except Exception as e:
        ok = False; print(f"  FAIL {m:18s} {type(e).__name__}: {e}")
import numpy, torch
if not numpy.__version__.startswith("1.23"): ok=False; print("  FAIL numpy is", numpy.__version__)
if not torch.cuda.is_available(): ok=False; print("  FAIL torch sees no GPU")
else: print("  GPU:", torch.cuda.get_device_name())
print("\n✅ environment OK" if ok else "\n❌ environment broken — send this output")
EOF

### 3a. Pretrained checkpoints
Official source is the Google Drive folder in GVHMR's `docs/INSTALL.md`; Drive folder downloads
often hit quota on Colab, so we pull the identical files from the public Hugging Face mirror
`camenduru/GVHMR` first and fall back to Drive. DPVO is skipped (not needed with `-s`, static tripod cam).

Target layout (from `docs/INSTALL.md`):
```
inputs/checkpoints/gvhmr/gvhmr_siga24_release.ckpt   (~163 MB)
inputs/checkpoints/hmr2/epoch=10-step=25000.ckpt     (~2.7 GB)
inputs/checkpoints/vitpose/vitpose-h-multi-coco.pth  (~2.5 GB)
inputs/checkpoints/yolo/yolov8x.pt                   (~137 MB)
```

In [ ]:
import os, shutil, subprocess
CK = "/content/GVHMR/inputs/checkpoints"
NEED = {  # path : minimum plausible size in bytes (catches HTML error pages / partial downloads)
    "gvhmr/gvhmr_siga24_release.ckpt": 150e6,
    "hmr2/epoch=10-step=25000.ckpt":   2.0e9,
    "vitpose/vitpose-h-multi-coco.pth": 2.0e9,
    "yolo/yolov8x.pt":                 100e6,
}
def missing():
    return [p for p, mn in NEED.items()
            if not os.path.exists(f"{CK}/{p}") or os.path.getsize(f"{CK}/{p}") < mn]

if missing():
    from huggingface_hub import hf_hub_download
    for p in missing():
        print("HF  ->", p)
        try:
            hf_hub_download("camenduru/GVHMR", p, local_dir=CK)
        except Exception as e:
            print("   HF failed:", e)

if missing():
    print("falling back to the official Google Drive folder ...")
    subprocess.run("pip install -q gdown && gdown --folder "
                   "https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD "
                   "-O /content/gdrive_ckpt --remaining-ok", shell=True)
    for p in missing():
        for root, _, files in os.walk("/content/gdrive_ckpt"):
            if os.path.basename(p) in files:
                os.makedirs(os.path.dirname(f"{CK}/{p}"), exist_ok=True)
                shutil.move(os.path.join(root, os.path.basename(p)), f"{CK}/{p}")

for p in NEED:
    f = f"{CK}/{p}"
    print(f"  {'ok ' if p not in missing() else 'BAD'} {p:40s} "
          f"{os.path.getsize(f)/1e6 if os.path.exists(f) else 0:8.1f} MB")
print("\n✅ checkpoints OK" if not missing() else "\n❌ still missing: " + ", ".join(missing()))

### 3b. SMPL + SMPL-X body models (needs your MPI logins)
Code expects exactly (checked in `hmr4d/utils/smplx_utils.py`):
```
inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl
inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz
```
This cell logs in to MPI's download server with the account you registered (prompted securely,
nothing is saved). If you registered both sites with the same email/password, enter it once.
Make sure you clicked **accept license** on both sites after registering, otherwise you get an HTML page instead of a zip.

**Manual fallback:** download `SMPL_python_v.1.1.0.zip` (smpl.is.tue.mpg.de → Downloads → "SMPL for Python")
and `models_smplx_v1_1.zip` (smpl-x.is.tue.mpg.de → Downloads → "SMPL-X v1.1"), upload them to `/content/`
with the file browser, then re-run this cell — it uses zips already in `/content/` without logging in.

In [ ]:
import os, glob, shutil, zipfile, getpass, requests
BM = "/content/GVHMR/inputs/checkpoints/body_models"
os.makedirs(f"{BM}/smpl", exist_ok=True); os.makedirs(f"{BM}/smplx", exist_ok=True)
SMPL_DST, SMPLX_DST = f"{BM}/smpl/SMPL_NEUTRAL.pkl", f"{BM}/smplx/SMPLX_NEUTRAL.npz"

def mpi_download(domain, sfile, out, user, pw):
    if os.path.exists(out) and zipfile.is_zipfile(out):
        print("using existing", out); return True
    url = f"https://download.is.tue.mpg.de/download.php?domain={domain}&sfile={sfile}&resume=1"
    with requests.post(url, data={"username": user, "password": pw}, stream=True, timeout=60) as r:
        if "text/html" in r.headers.get("content-type", ""):
            print(f"❌ {domain}: got a web page, not a zip -> wrong login or license not accepted"); return False
        with open(out, "wb") as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
    ok = zipfile.is_zipfile(out); print(domain, "downloaded", os.path.getsize(out)//1_000_000, "MB", "" if ok else "(NOT a zip!)")
    return ok

need_login = not (os.path.exists(SMPL_DST) and os.path.exists(SMPLX_DST))
have_zips = zipfile.is_zipfile("/content/SMPL_python_v.1.1.0.zip") if os.path.exists("/content/SMPL_python_v.1.1.0.zip") else False
have_zips &= zipfile.is_zipfile("/content/models_smplx_v1_1.zip") if os.path.exists("/content/models_smplx_v1_1.zip") else False
if need_login and not have_zips:
    u = input("MPI email: "); p = getpass.getpass("MPI password: ")
else:
    u = p = None

if not os.path.exists(SMPL_DST):
    z = "/content/SMPL_python_v.1.1.0.zip"
    if mpi_download("smpl", "SMPL_python_v.1.1.0.zip", z, u, p):
        with zipfile.ZipFile(z) as zf:
            name = [n for n in zf.namelist() if n.endswith("basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl")][0]
            with zf.open(name) as s, open(SMPL_DST, "wb") as d: shutil.copyfileobj(s, d)

if not os.path.exists(SMPLX_DST):
    z = "/content/models_smplx_v1_1.zip"
    if mpi_download("smplx", "models_smplx_v1_1.zip", z, u, p):
        with zipfile.ZipFile(z) as zf:
            name = [n for n in zf.namelist() if n.endswith("SMPLX_NEUTRAL.npz")][0]
            with zf.open(name) as s, open(SMPLX_DST, "wb") as d: shutil.copyfileobj(s, d)
del p

In [ ]:
%%bash
# Verify body models actually load (in the venv, where chumpy works for the SMPL .pkl)
export MPLBACKEND=Agg PYTHONPATH=/content/GVHMR   # Colab's inline matplotlib backend doesn't exist in the venv
cd /content/GVHMR && /content/gvenv/bin/python - <<'EOF'
ok = True
try:
    from hmr4d.utils.smplx_utils import make_smplx
    m = make_smplx("supermotion"); print("  ok  SMPL-X (supermotion) loaded")
    s = make_smplx("smpl");        print("  ok  SMPL loaded, faces", s.faces.shape)
except Exception as e:
    ok = False; import traceback; traceback.print_exc()
print("\n✅ body models OK" if ok else "\n❌ body models broken — send this output")
EOF

### (Optional) Cache everything on Google Drive
Colab wipes `/content` every session. Run this once after 3a/3b succeed to save the ~5.5 GB of weights
to Drive; next session, run it again *before* 3a/3b and it restores them (3a/3b then skip downloading).

In [ ]:
from google.colab import drive; import os, subprocess
drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/gvhmr_cache/checkpoints"
CK = "/content/GVHMR/inputs/checkpoints"
os.makedirs(CK, exist_ok=True)
if os.path.isdir(CACHE) and os.listdir(CACHE):
    print("restoring from Drive ..."); subprocess.run(f'cp -rn "{CACHE}/." "{CK}/"', shell=True)
else:
    print("saving to Drive ...");      os.makedirs(CACHE, exist_ok=True)
    subprocess.run(f'cp -rn "{CK}/." "{CACHE}/"', shell=True)
print("done")

### 4. Upload your clip

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('/content/GVHMR/inputs/videos', exist_ok=True)
up = files.upload()  # pick your clip
name = list(up.keys())[0]
dst = '/content/GVHMR/inputs/videos/clip.mp4'
shutil.move(name, dst)
print('saved to', dst)

### 5. Run GVHMR
`-s` = static camera (tripod) → skips visual odometry. Outputs go to `outputs/demo/clip/`.

In [ ]:
!cd /content/GVHMR && MPLBACKEND=Agg PYTHONPATH=/content/GVHMR /content/gvenv/bin/python tools/demo/demo.py --video=inputs/videos/clip.mp4 -s

In [ ]:
import os
OUT = "/content/GVHMR/outputs/demo/clip"
expect = ["hmr4d_results.pt", "1_incam.mp4", "2_global.mp4", "clip_3_incam_global_horiz.mp4"]
for f in expect:
    p = f"{OUT}/{f}"
    print(f"  {'ok ' if os.path.exists(p) else 'MISSING'} {f:32s} {os.path.getsize(p)/1e6 if os.path.exists(p) else 0:7.1f} MB")
print("\n✅ GVHMR run complete" if all(os.path.exists(f"{OUT}/{f}") for f in expect) else "\n❌ demo did not finish — send the output of the cell above")

### 6. Watch the overlay preview
Left = mesh over your video (camera view), right = the motion in world space. **Check this before converting** —
if the capture is bad here, re-shoot.

In [ ]:
from IPython.display import Video
Video("/content/GVHMR/outputs/demo/clip/clip_3_incam_global_horiz.mp4", embed=True, width=900)

### 7. GVHMR → Roblox animation (.rbxmx)
Converts `hmr4d_results.pt` straight into an R15 `KeyframeSequence`. No Blender.
In Studio: Explorer → right-click **Workspace** → **Insert from File…** → pick the downloaded file,
then play it with `PlayTest.server.lua` (set `KS_NAME` to the name below).

In [ ]:
%%writefile /content/smpl_to_r15.py
"""
smpl_to_r15.py — GVHMR (SMPL/SMPL-X body pose) -> Roblox R15 KeyframeSequence (.rbxmx)

No Blender. In Studio: Explorer > right-click Workspace > Insert from File... > pick the .rbxmx.

  python smpl_to_r15.py hmr4d_results.pt  out.rbxmx  [--name Thanos_Shot12] [--fps-step 1]
  python smpl_to_r15.py --test            test.rbxmx      # synthetic convention check

How it works
  SMPL stores each joint's rotation relative to its parent, with the rest pose = T-pose,
  Y-up, facing +Z. We run forward kinematics to get every joint's world rotation, convert
  to Roblox space (Y-up, facing -Z = a 180° turn about Y), map SMPL joints onto R15 parts,
  correct for R15's arms-down rest pose, and store Motor6D.Transform per part:
      Transform(part) = World(parent)^-1 * World(part)
  This assumes the rig's Motor6D C0/C1 have no rotation (true for Studio's Rig Builder R15).
"""
import sys, argparse
import numpy as np
from scipy.spatial.transform import Rotation as R

# first 22 SMPL / SMPL-X body joints
SMPL_PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19]

# R15 part -> (R15 parent part, SMPL joint whose WORLD rotation drives it)
# Spine1-3 merge into the Waist (UpperTorso), Neck+Head into Head, Collar into UpperArm.
R15 = {
    "LowerTorso":    ("HumanoidRootPart", 0),   # Pelvis
    "UpperTorso":    ("LowerTorso", 9),         # Spine3
    "Head":          ("UpperTorso", 15),        # Head
    "LeftUpperArm":  ("UpperTorso", 16),        # L_Shoulder
    "LeftLowerArm":  ("LeftUpperArm", 18),      # L_Elbow
    "LeftHand":      ("LeftLowerArm", 20),      # L_Wrist
    "RightUpperArm": ("UpperTorso", 17),
    "RightLowerArm": ("RightUpperArm", 19),
    "RightHand":     ("RightLowerArm", 21),
    "LeftUpperLeg":  ("LowerTorso", 1),         # L_Hip
    "LeftLowerLeg":  ("LeftUpperLeg", 4),       # L_Knee
    "LeftFoot":      ("LeftLowerLeg", 7),       # L_Ankle
    "RightUpperLeg": ("LowerTorso", 2),
    "RightLowerLeg": ("RightUpperLeg", 5),
    "RightFoot":     ("RightLowerLeg", 8),
}

# SMPL -> Roblox axes: rotate 180° about Y (SMPL faces +Z, Roblox faces -Z)
M = np.diag([-1.0, 1.0, -1.0])

# Rest-pose correction: R15 arms hang down (-Y); SMPL rest is a T-pose.
# In Roblox axes the character's left is -X, right is +X.
_rz = lambda deg: R.from_euler("z", deg, degrees=True).as_matrix()
REST_FIX = {p: np.eye(3) for p in R15}
for p in ("LeftUpperArm", "LeftLowerArm", "LeftHand"):    REST_FIX[p] = _rz(-90)   # -Y -> -X
for p in ("RightUpperArm", "RightLowerArm", "RightHand"): REST_FIX[p] = _rz(+90)   # -Y -> +X

STUDS_PER_METER = 3.2   # R15 hip height (~3 studs) / human pelvis height (~0.93 m); tweak per character


def world_rotations(pose):
    """pose (N,22,3) axis-angle local rotations -> (N,22,3,3) world rotations in SMPL axes."""
    N = len(pose)
    local = R.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(N, 22, 3, 3)
    world = np.empty_like(local)
    for j, p in enumerate(SMPL_PARENTS):
        world[:, j] = local[:, j] if p < 0 else world[:, p] @ local[:, j]
    return world


def to_r15_transforms(global_orient, body_pose, transl):
    """Returns {part: (N,4,4)} Motor6D.Transform matrices, Roblox space, studs."""
    N = len(global_orient)
    pose = np.concatenate([global_orient.reshape(N, 1, 3), body_pose.reshape(N, 21, 3)], 1)
    W = M @ world_rotations(pose) @ M.T                       # world rotations in Roblox axes
    G = {"HumanoidRootPart": np.broadcast_to(np.eye(3), (N, 3, 3))}
    for part, (_, j) in R15.items():
        G[part] = W[:, j] @ REST_FIX[part]
    out = {}
    for part, (parent, _) in R15.items():
        T = np.tile(np.eye(4), (N, 1, 1))
        T[:, :3, :3] = np.swapaxes(G[parent], 1, 2) @ G[part]
        out[part] = T
    # root motion: pelvis displacement from frame 0 (pelvis = transl + constant rest offset)
    d = (transl.reshape(N, 3) - transl.reshape(N, 3)[0]) @ M.T * STUDS_PER_METER
    out["LowerTorso"][:, :3, 3] = d
    return out


# ------------------------------------------------------------------ .rbxmx writer
CHILDREN = {}
for part, (parent, _) in R15.items():
    CHILDREN.setdefault(parent, []).append(part)

def _cframe(T):
    x, y, z = T[:3, 3]; r = T[:3, :3].reshape(-1)
    tags = ["X", "Y", "Z"] + [f"R{i}{j}" for i in range(3) for j in range(3)]
    return "".join(f"<{t}>{v:.6f}</{t}>" for t, v in zip(tags, [x, y, z, *r]))

def write_rbxmx(transforms, path, name="GVHMR_Anim", fps=30.0, step=1, loop=False):
    N = len(next(iter(transforms.values())))
    ref = iter(range(10**9))
    out = ['<roblox xmlns:xmime="http://www.w3.org/2005/05/xmlmime" '
           'xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="4">',
           f'<Item class="KeyframeSequence" referent="RBX{next(ref)}"><Properties>'
           f'<string name="Name">{name}</string><bool name="Loop">{str(loop).lower()}</bool>'
           f'<token name="Priority">2</token></Properties>']   # Priority 2 = Action

    def pose(part, f):
        T = transforms[part][f] if part in transforms else np.eye(4)
        s = (f'<Item class="Pose" referent="RBX{next(ref)}"><Properties>'
             f'<CoordinateFrame name="CFrame">{_cframe(T)}</CoordinateFrame>'
             f'<token name="EasingDirection">0</token><token name="EasingStyle">0</token>'
             f'<string name="Name">{part}</string><float name="Weight">1</float></Properties>')
        return s + "".join(pose(c, f) for c in CHILDREN.get(part, [])) + "</Item>"

    for f in list(range(0, N, step)) + ([N - 1] if (N - 1) % step else []):
        out.append(f'<Item class="Keyframe" referent="RBX{next(ref)}"><Properties>'
                   f'<string name="Name">Keyframe</string><float name="Time">{f / fps:.5f}</float>'
                   f'</Properties>{pose("HumanoidRootPart", f)}</Item>')
    out.append("</Item></roblox>")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print(f"wrote {path}: {N} frames ({N / fps:.2f}s), {len(range(0, N, step))} keyframes")


# ------------------------------------------------------------------ inputs
def load_gvhmr(path):
    import torch
    P = torch.load(path, map_location="cpu")["smpl_params_global"]
    g = lambda k: P[k].detach().cpu().numpy()
    return g("global_orient"), g("body_pose"), g("transl")

def test_motion(fps=30):
    """5 one-second phases with known expected results (see README)."""
    N = 5 * fps; go = np.zeros((N, 3)); bp = np.zeros((N, 21, 3)); tr = np.zeros((N, 3))
    J = lambda smpl_idx: smpl_idx - 1           # body_pose index of an SMPL joint
    ramp = lambda f, a: np.clip((f - a * fps) / (0.5 * fps), 0, 1)   # 0->1 over half a second
    from scipy.spatial.transform import Slerp
    r_down, r_fwd = R.from_euler("z", 90, degrees=True), R.from_euler("y", 90, degrees=True)
    punch_path = Slerp([0, 1], R.concatenate([r_down, r_fwd]))   # SMPL right arm: -Y -> +Z
    for f in range(N):
        down = ramp(f, 1)                                  # 1s: arms down to sides
        bp[f, J(16)] = [0, 0, -np.pi / 2 * down]           # L_Shoulder: SMPL +X -> -Y
        bp[f, J(17)] = [0, 0, +np.pi / 2 * down]           # R_Shoulder: SMPL -X -> -Y
        punch = ramp(f, 2)                                 # 2s: RIGHT arm punches forward
        if punch > 0:
            bp[f, J(17)] = punch_path(punch).as_rotvec()
        knee = ramp(f, 3)                                  # 3s: LEFT knee lifts forward
        bp[f, J(1)] = [-np.pi / 2 * knee, 0, 0]            # L_Hip flex
        bp[f, J(4)] = [np.pi / 2 * knee, 0, 0]             # L_Knee bend back
        turn = ramp(f, 4)                                  # 4s: body turns to ITS left + steps up 1 m
        go[f] = [0, np.pi / 2 * turn, 0]
        tr[f] = [0, 1.0 * turn, 0]
    return go, bp, tr


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("inp", nargs="?"); ap.add_argument("out", nargs="?")
    ap.add_argument("--test", metavar="OUT"); ap.add_argument("--name")
    ap.add_argument("--fps-step", type=int, default=1, help="keep every Nth frame (1 = all 30 fps)")
    a = ap.parse_args()
    if a.test:
        write_rbxmx(to_r15_transforms(*test_motion()), a.test, a.name or "GVHMR_Test")
    else:
        write_rbxmx(to_r15_transforms(*load_gvhmr(a.inp)), a.out, a.name or "GVHMR_Anim", step=a.fps_step)

In [ ]:
SHOT = "clip"                 # the output folder name under outputs/demo/
NAME = "Shot_" + SHOT         # KeyframeSequence name inside Roblox
!cd /content/GVHMR && MPLBACKEND=Agg PYTHONPATH=/content/GVHMR /content/gvenv/bin/python /content/smpl_to_r15.py outputs/demo/{SHOT}/hmr4d_results.pt /content/{NAME}.rbxmx --name {NAME}
from google.colab import files
files.download(f"/content/{NAME}.rbxmx")